# Day 3 — pandas Workflow and Data-Quality Report

**Goal:** Load, inspect, filter, group, and validate text data with pandas.

**Required evidence:** Produce a compact data-quality report.

## 1. Import pandas

In [1]:
from io import StringIO
import re

import pandas as pd

print('pandas version:', pd.__version__)

pandas version: 2.2.2


## 2. Load a small text dataset

`pd.read_csv()` is used here with an in-memory CSV so the notebook is self-contained. In a real project, replace `StringIO(csv_data)` with a file path such as `../data/posts.csv`.

In [2]:
csv_data = '''id,platform,text,label
1,YouTube,Breaking football news!,2
2,Facebook,Football news update,1
3,News,Election result announced,0
4,YouTube,You will not believe this!!!,3
5,Facebook,,1
6,News,   ,0
7,YouTube,Breaking football news!,2
8,Instagram,Unexpected celebrity secret,4
9,Facebook,Click now to discover the truth,3
10,News,Government publishes annual budget,0
'''

df = pd.read_csv(StringIO(csv_data))
df

,id,platform,text,label
0,1,YouTube,Breaking football news!,2
1,2,Facebook,Football news update,1
2,3,News,Election result announced,0
3,4,YouTube,You will not believe this!!!,3
4,5,Facebook,NaN,1
5,6,News,,0
6,7,YouTube,Breaking football news!,2
7,8,Instagram,Unexpected celebrity secret,4
8,9,Facebook,Click now to discover the truth,3
9,10,News,Government publishes annual budget,0


## 3. Inspect the dataset

Important first checks are `head()`, `shape`, column names, data types, and missing values.

In [3]:
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
print('\nData types:')
print(df.dtypes)
print('\nMissing values:')
print(df.isna().sum())

df.head()

Shape: (10, 4)
Columns: ['id', 'platform', 'text', 'label']

Data types:
id           int64
platform    object
text        object
label        int64
dtype: object

Missing values:
id          0
platform    0
text        1
label       0
dtype: int64


,id,platform,text,label
0,1,YouTube,Breaking football news!,2
1,2,Facebook,Football news update,1
2,3,News,Election result announced,0
3,4,YouTube,You will not believe this!!!,3
4,5,Facebook,NaN,1


## 4. Prepare validation columns

A whitespace-only string is not technically missing to pandas, so we normalize whitespace before checking empty text. The original `text` column remains unchanged.

In [4]:
def normalize_for_validation(value):
    if pd.isna(value):
        return ''
    return re.sub(r'\s+', ' ', str(value)).strip()

work_df = df.copy()
work_df['normalized_text'] = work_df['text'].apply(normalize_for_validation)
work_df['text_length'] = work_df['normalized_text'].str.len()
work_df['is_empty_text'] = work_df['normalized_text'].eq('')
work_df['is_duplicate_text'] = work_df.duplicated(
    subset='normalized_text', keep=False
) & ~work_df['is_empty_text']

work_df[['id', 'text', 'normalized_text', 'text_length',
         'is_empty_text', 'is_duplicate_text']]

,id,text,normalized_text,text_length,is_empty_text,is_duplicate_text
0,1,Breaking football news!,Breaking football news!,23,False,True
1,2,Football news update,Football news update,20,False,False
2,3,Election result announced,Election result announced,25,False,False
3,4,You will not believe this!!!,You will not believe this!!!,28,False,False
4,5,NaN,,0,True,False
5,6,,,0,True,False
6,7,Breaking football news!,Breaking football news!,23,False,True
7,8,Unexpected celebrity secret,Unexpected celebrity secret,27,False,False
8,9,Click now to discover the truth,Click now to discover the truth,31,False,False
9,10,Government publishes annual budget,Government publishes annual budget,34,False,False


## 5. Filter rows

Filtering uses Boolean conditions. This example selects usable rows with non-empty text and labels from 0 through 3.

In [5]:
valid_label_mask = work_df['label'].between(0, 3)
usable_mask = ~work_df['is_empty_text'] & valid_label_mask
usable_df = work_df.loc[usable_mask].copy()

print('Usable rows:', len(usable_df))
usable_df[['id', 'platform', 'normalized_text', 'label']]

Usable rows: 7


,id,platform,normalized_text,label
0,1,YouTube,Breaking football news!,2
1,2,Facebook,Football news update,1
2,3,News,Election result announced,0
3,4,YouTube,You will not believe this!!!,3
6,7,YouTube,Breaking football news!,2
8,9,Facebook,Click now to discover the truth,3
9,10,News,Government publishes annual budget,0


## 6. Group and summarize

`groupby()` helps reveal class and platform distributions.

In [6]:
label_summary = (
    usable_df.groupby('label')
    .agg(row_count=('id', 'count'), average_text_length=('text_length', 'mean'))
    .round(2)
)

platform_summary = (
    usable_df.groupby('platform')
    .size()
    .rename('row_count')
    .sort_values(ascending=False)
)

print('Label summary:')
display(label_summary)
print('Platform summary:')
display(platform_summary.to_frame())

Label summary:


,row_count,average_text_length
label,,
0,2,29.5
1,1,20.0
2,2,23.0
3,2,29.5


Platform summary:


,row_count
platform,
YouTube,3
Facebook,2
News,2


## 7. Validate the data

For this exercise, valid platforms are YouTube, Facebook, and News; valid labels are 0–3. Duplicate rows are counted as rows participating in a duplicate group.

In [7]:
allowed_platforms = {'YouTube', 'Facebook', 'News'}

missing_text_count = int(work_df['is_empty_text'].sum())
duplicate_text_row_count = int(work_df['is_duplicate_text'].sum())
invalid_label_count = int((~work_df['label'].between(0, 3)).sum())
invalid_platform_count = int((~work_df['platform'].isin(allowed_platforms)).sum())

invalid_rows = work_df.loc[
    work_df['is_empty_text']
    | work_df['is_duplicate_text']
    | ~work_df['label'].between(0, 3)
    | ~work_df['platform'].isin(allowed_platforms),
    ['id', 'platform', 'text', 'label', 'is_empty_text', 'is_duplicate_text'],
]

invalid_rows

,id,platform,text,label,is_empty_text,is_duplicate_text
0,1,YouTube,Breaking football news!,2,False,True
4,5,Facebook,NaN,1,True,False
5,6,News,,0,True,False
6,7,YouTube,Breaking football news!,2,False,True
7,8,Instagram,Unexpected celebrity secret,4,False,False


## 8. Required evidence — compact data-quality report

Each check reports a count and a simple status.

In [8]:
report_rows = [
    ('Total rows', len(work_df), 'INFO'),
    ('Total columns', work_df.shape[1], 'INFO'),
    ('Empty or missing text', missing_text_count,
     'PASS' if missing_text_count == 0 else 'REVIEW'),
    ('Rows in duplicate-text groups', duplicate_text_row_count,
     'PASS' if duplicate_text_row_count == 0 else 'REVIEW'),
    ('Invalid labels', invalid_label_count,
     'PASS' if invalid_label_count == 0 else 'REVIEW'),
    ('Unexpected platforms', invalid_platform_count,
     'PASS' if invalid_platform_count == 0 else 'REVIEW'),
    ('Usable rows', int(usable_mask.sum()), 'INFO'),
]

quality_report = pd.DataFrame(
    report_rows, columns=['check', 'count', 'status']
)
quality_report

,check,count,status
0,Total rows,10,INFO
1,Total columns,8,INFO
2,Empty or missing text,2,REVIEW
3,Rows in duplicate-text groups,2,REVIEW
4,Invalid labels,1,REVIEW
5,Unexpected platforms,1,REVIEW
6,Usable rows,7,INFO


## 9. Verification

Assertions make the expected results explicit and detect accidental changes.

In [9]:
assert len(work_df) == 10
assert missing_text_count == 2
assert duplicate_text_row_count == 2
assert invalid_label_count == 1
assert invalid_platform_count == 1
assert int(usable_mask.sum()) == 7
assert set(quality_report.columns) == {'check', 'count', 'status'}

print('All Day 3 pandas and data-quality checks passed successfully!')

All Day 3 pandas and data-quality checks passed successfully!


## Independent practice

1. Add one new valid record and rerun the notebook.
2. Filter only records with `label >= 2`.
3. Group the usable data by both `platform` and `label`.
4. Add a report check for texts shorter than 10 characters.
5. Replace the sample CSV with your own project dataset when it becomes available.